# Vision service — endpoint check

Run this **from the Pi**, against the vision service on the Mac mini. It pings
every endpoint, verifies the response contract, measures latency, and exercises
the error paths.

This is the diagnostic for **Phase V1** in `docs/VISION-DESIGN.md` — the phase
whose whole job is to measure real bandwidth and p99 latency *before* anything is
built on top of them. Every constant in that document's appendix is a guess until
this notebook has run on your actual wifi.

## Before you start

```bash
# on the Mac mini
cd vision_service && ./run.sh

# on the Pi
pip install requests pillow notebook
jupyter notebook tests/hardware/check_vision_service.ipynb
```

Set `BASE_URL` in the next cell to the mini's address. Then **Run All**.

Nothing here writes to the robot or moves anything. The only side effect is a
temporary enrollment in the face gallery, which the notebook removes when it
finishes.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
import os

# The mini. Try, in order: an env var you set, then this literal.
# Find the address on the mini with:  ipconfig getifaddr en0
BASE_URL = os.environ.get("VISION_SERVICE_BASE_URL", "http://192.168.1.50:8080").rstrip("/")

# A real face photo makes the identity endpoints meaningful. Point this at a
# JPEG of someone's face, or leave it None to skip those checks.
FACE_IMAGE = os.environ.get("VISION_TEST_FACE", None)   # e.g. "~/faces/roshan.jpg"

# Latency benchmark. 40 calls at 2 Hz is 20 seconds — long enough for a p95 to
# mean something, short enough that you will actually wait for it.
BENCH_CALLS = 40

# Must match the Pi-side constants in robot_core/settings.py.
DETECT_TIMEOUT_S = 0.6          # hard abandon, no retry
DETECT_HZ = 2.0
DETECT_LONG_EDGE_PX = 640

print(f"target: {BASE_URL}")
print(f"face image: {FACE_IMAGE or '(none — identity checks will be skipped)'}")

In [ ]:
# ---------------------------------------------------------------------------
# Harness. Checks record their outcome instead of raising, so one failure does
# not stop the run — you want the whole picture in one pass, not one error at a
# time.
# ---------------------------------------------------------------------------
import io, json, statistics, time
from concurrent.futures import ThreadPoolExecutor

import requests

RESULTS = []          # (section, name, status, detail)   status: PASS | FAIL | SKIP | INFO

def record(section, name, status, detail=""):
    RESULTS.append((section, name, status, detail))
    mark = {"PASS": "  ok  ", "FAIL": " FAIL ", "SKIP": " skip ", "INFO": " info "}[status]
    print(f"[{mark}] {name}" + (f"\n         {detail}" if detail else ""))
    return status == "PASS"

def check(section, name, ok, detail=""):
    return record(section, name, "PASS" if ok else "FAIL", detail)

def skip(section, name, why):
    return record(section, name, "SKIP", why)

def info(section, name, detail):
    return record(section, name, "INFO", detail)

# One Session for the whole notebook: keep-alive matters at 2 Hz, and it is what
# the real gateway client does, so the numbers below reflect production.
S = requests.Session()
S.headers["Connection"] = "keep-alive"

def url(path):
    return f"{BASE_URL}{path}"

def post_frame(path, jpeg, *, field="image", filename="frame.jpg", timeout=10.0, **form):
    files = {field: (filename, jpeg, "image/jpeg")}
    data = {k: str(v) for k, v in form.items()}
    return S.post(url(path), files=files, data=data, timeout=timeout)

print("harness ready")

---
## 1 · Is it there at all?

If this fails, nothing below will work. The usual causes, in the order they
actually happen: the mini went to sleep, the service is not running, or
`BASE_URL` has the wrong address.

In [ ]:
SECTION = "reachability"
health = None

try:
    r = S.get(url("/healthz"), timeout=5.0)
    health = r.json()
    check(SECTION, "GET /healthz responds", r.status_code == 200, f"HTTP {r.status_code}")
    check(SECTION, "reports healthy", health.get("status") == "ok", json.dumps(health, indent=2))

    # The single most important field. "fake" means the service is inventing
    # objects — every detection below would be meaningless.
    backend = health.get("backend")
    if backend == "real":
        check(SECTION, "backend is real", True, f"versions: {health.get('versions')}")
    else:
        check(SECTION, "backend is real", False,
              f"backend={backend!r} — the service is SERVING FAKE DETECTIONS. "
              f"Start it with VS_BACKEND=real. Contract checks below still mean "
              f"something; detections do not.")

    info(SECTION, "uptime", f"{health.get('uptime_s')} s, queue_depth={health.get('queue_depth')}")
    info(SECTION, "gallery", json.dumps(health.get("gallery", {})))

except requests.exceptions.ConnectTimeout:
    check(SECTION, "GET /healthz responds", False,
          f"timed out connecting to {BASE_URL}. The mini is likely asleep — a "
          f"sleeping mini looks exactly like bad wifi from here. Check caffeinate "
          f"is running (it is in run.sh).")
except requests.exceptions.ConnectionError as exc:
    check(SECTION, "GET /healthz responds", False,
          f"cannot reach {BASE_URL}: {exc.__class__.__name__}. Check (1) the "
          f"service is running on the mini, (2) BASE_URL has the right address, "
          f"(3) you are on the same network.")
except Exception as exc:
    check(SECTION, "GET /healthz responds", False, f"{type(exc).__name__}: {exc}")

REACHABLE = health is not None and health.get("status") == "ok"
REAL_BACKEND = bool(health and health.get("backend") == "real")

In [ ]:
SECTION = "models"
if REACHABLE:
    m = S.get(url("/v1/models"), timeout=5.0).json()
    check(SECTION, "GET /v1/models", set(m.get("versions", {})) == {"detect", "face", "depth"},
          json.dumps(m, indent=2))
    # The Pi must branch on this flag, never on the depths_m field name.
    if not m.get("depth_is_metric", False):
        info(SECTION, "depth is NOT metric",
             "the depth model returns relative inverse depth — values order "
             "correctly but are not metres. Prefer lidar range at bearing.")
    DEPTH_IS_METRIC = m.get("depth_is_metric", False)
else:
    skip(SECTION, "GET /v1/models", "service unreachable")
    DEPTH_IS_METRIC = False

---
## 2 · A frame to send

Prefers the real camera. Falls back to a synthetic frame so the contract checks
still run on a Pi with no camera attached.

**With the real backend, a synthetic frame legitimately returns zero
detections** — a flat grey rectangle contains no objects. That is correct
behaviour, not a failure. Point the camera at a room to get a meaningful answer.

In [ ]:
SECTION = "frame"
FRAME = None
FRAME_SOURCE = None

# --- 1. the real camera -----------------------------------------------------
try:
    from picamera2 import Picamera2

    picam = Picamera2()
    cfg = picam.create_still_configuration(main={"size": (DETECT_LONG_EDGE_PX,
                                                         int(DETECT_LONG_EDGE_PX * 9 / 16))})
    picam.configure(cfg)
    picam.start()
    time.sleep(1.5)                      # let auto-exposure settle
    buf = io.BytesIO()
    picam.capture_file(buf, format="jpeg")
    picam.stop(); picam.close()
    FRAME = buf.getvalue()
    FRAME_SOURCE = "picamera2"
except Exception as exc:
    info(SECTION, "camera unavailable", f"{type(exc).__name__}: {exc}")

# --- 2. synthetic -----------------------------------------------------------
if FRAME is None:
    try:
        from PIL import Image, ImageDraw

        img = Image.new("RGB", (DETECT_LONG_EDGE_PX, 360), (70, 90, 110))
        d = ImageDraw.Draw(img)
        d.rectangle([80, 60, 260, 300], fill=(150, 140, 130))
        d.ellipse([140, 20, 210, 90], fill=(200, 180, 170))
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=75)
        FRAME = buf.getvalue()
        FRAME_SOURCE = "synthetic"
    except ImportError:
        pass

if FRAME:
    kb = len(FRAME) / 1024
    check(SECTION, f"frame acquired ({FRAME_SOURCE})", True, f"{kb:.1f} KB")
    # Payload sizing is a standing rule, not a detail: this shares a radio with a
    # continuous audio uplink that has already died once from contention.
    if kb > 60:
        check(SECTION, "payload within budget", False,
              f"{kb:.1f} KB is large. At {DETECT_HZ} Hz that is "
              f"{kb * DETECT_HZ:.0f} KB/s competing with the voice session. "
              f"Target ~30 KB — check DETECT_LONG_EDGE_PX and JPEG quality.")
    else:
        check(SECTION, "payload within budget", True,
              f"{kb * DETECT_HZ:.0f} KB/s at {DETECT_HZ} Hz")
    if FRAME_SOURCE == "synthetic" and REAL_BACKEND:
        info(SECTION, "synthetic frame + real models",
             "expect zero detections — there is nothing in this image. Contract "
             "checks still apply; object counts do not.")
else:
    check(SECTION, "frame acquired", False, "no picamera2 and no Pillow — pip install pillow")

---
## 3 · `POST /v1/detect`

The hot path. What matters here is not *what* it found but that the response is
placeable in time and space: the `seq` you sent comes back, and `image_size`
tells you what pixel space the boxes are in.

In [ ]:
SECTION = "detect"
SEQ = 10_000

if REACHABLE and FRAME:
    SEQ += 1
    t0 = time.perf_counter()
    r = post_frame("/v1/detect", FRAME, seq=SEQ, min_confidence=0.4)
    rtt_ms = (time.perf_counter() - t0) * 1000

    if check(SECTION, "HTTP 200", r.status_code == 200, f"HTTP {r.status_code}: {r.text[:200]}"):
        b = r.json()

        # The contract. Without the seq echo the Pi cannot look up the capture
        # time, gimbal angles or odom pose for this frame — a result it cannot
        # place in time is worse than no result.
        check(SECTION, "seq echoed", b.get("seq") == SEQ, f"sent {SEQ}, got {b.get('seq')}")
        check(SECTION, "model_version present", bool(b.get("model_version")), b.get("model_version"))
        check(SECTION, "image_size echoed", len(b.get("image_size", [])) == 2, str(b.get("image_size")))

        w, h = b["image_size"]
        bad = []
        for d in b.get("detections", []):
            if not {"class", "confidence", "bbox"} <= set(d):
                bad.append(f"missing keys: {d}")
                continue
            x1, y1, x2, y2 = d["bbox"]
            if not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h):
                bad.append(f"{d['class']} bbox {d['bbox']} outside {w}x{h}")
            if not 0.0 <= d["confidence"] <= 1.0:
                bad.append(f"{d['class']} confidence {d['confidence']} out of range")
        check(SECTION, "every bbox is well-formed and inside the image", not bad, "; ".join(bad))

        # A bbox in the wrong pixel space looks exactly like bad camera
        # calibration, and you will spend an evening on it.
        check(SECTION, "image_size matches what we sent", [w, h] != [0, 0],
              f"server received {w}x{h}")

        server_ms = b.get("inference_ms", 0)
        info(SECTION, "timing",
             f"round trip {rtt_ms:.0f} ms = {server_ms:.0f} ms inference "
             f"+ {rtt_ms - server_ms:.0f} ms network/encode")
        info(SECTION, "detections",
             ", ".join(f"{d['class']}({d['confidence']:.2f})" for d in b["detections"]) or "(none)")
        LAST_DETECT = b
else:
    skip(SECTION, "POST /v1/detect", "service unreachable or no frame")
    LAST_DETECT = None

---
## 4 · Latency — the number this notebook exists for

`stopping distance = speed × p99` is the lidar's problem, but perception has the
same shape: every constant in VISION-DESIGN.md assumes a p99 detect latency
around 300 ms. Measure it; don't estimate it.

> **Run this again with the voice session live and the robot driving.** A
> latency measured on an idle Pi and a quiet radio tells you nothing about the
> moment it matters. That is the whole lesson of `HARDWARE-BASICS.md` Part 2.

In [ ]:
SECTION = "latency"

if REACHABLE and FRAME:
    rtts, server_times, failures = [], [], 0
    period = 1.0 / DETECT_HZ

    print(f"sending {BENCH_CALLS} calls at {DETECT_HZ} Hz "
          f"(~{BENCH_CALLS / DETECT_HZ:.0f}s)...", flush=True)
    for i in range(BENCH_CALLS):
        started = time.perf_counter()
        SEQ += 1
        try:
            r = post_frame("/v1/detect", FRAME, seq=SEQ, timeout=DETECT_TIMEOUT_S * 4)
            elapsed_ms = (time.perf_counter() - started) * 1000
            if r.status_code == 200:
                rtts.append(elapsed_ms)
                server_times.append(r.json().get("inference_ms", 0.0))
            else:
                failures += 1
        except requests.exceptions.RequestException:
            failures += 1
        # Pace to the real call rate rather than hammering — a burst measures
        # something the robot will never do.
        time.sleep(max(0.0, period - (time.perf_counter() - started)))

    if rtts:
        rtts.sort()
        p = lambda q: rtts[min(int(len(rtts) * q), len(rtts) - 1)]
        p50, p95, p99 = p(0.50), p(0.95), p(0.99)
        net = statistics.median(rtts) - statistics.median(server_times)

        print()
        print(f"  calls        {len(rtts)} ok, {failures} failed")
        print(f"  p50          {p50:6.0f} ms")
        print(f"  p95          {p95:6.0f} ms")
        print(f"  p99          {p99:6.0f} ms   <-- the one that matters")
        print(f"  max          {rtts[-1]:6.0f} ms")
        print(f"  inference    {statistics.median(server_times):6.0f} ms (median, server-reported)")
        print(f"  network      {net:6.0f} ms (median round trip minus inference)")
        print(f"  bandwidth    {len(FRAME) / 1024 * DETECT_HZ:6.0f} KB/s at {DETECT_HZ} Hz")

        check(SECTION, "no failed calls", failures == 0, f"{failures} of {BENCH_CALLS} failed")

        budget = DETECT_TIMEOUT_S * 1000
        within = p99 < budget
        detail = (f"p99 {p99:.0f} ms against a {budget:.0f} ms budget — "
                  f"{budget - p99:.0f} ms of headroom"
                  if within else
                  f"p99 {p99:.0f} ms exceeds the {budget:.0f} ms budget, so roughly "
                  f"1 call in 100 is abandoned and thrown away. Reduce the payload "
                  f"first, then consider raising DETECT_TIMEOUT_S.")
        check(SECTION, f"p99 within DETECT_TIMEOUT_S ({budget:.0f} ms)", within, detail)

        if p99 > 300:
            info(SECTION, "p99 above the design assumption",
                 f"{p99:.0f} ms vs the ~300 ms VISION-DESIGN.md assumes. The "
                 f"tracker absorbs some of this, but revisit MAX_RESULT_AGE_S "
                 f"and the follow-behavior margins before building on top.")

        # A tail far above the median means contention, not slowness — and
        # contention is what kills the audio session.
        if p99 > p50 * 3:
            info(SECTION, "tail is long relative to median",
                 f"p99/p50 = {p99 / p50:.1f}x. Something is competing for the "
                 f"radio or the CPU. Check what else is running.")
        BENCH = {"p50": p50, "p95": p95, "p99": p99, "failures": failures}
    else:
        check(SECTION, "benchmark completed", False, f"all {BENCH_CALLS} calls failed")
        BENCH = None
else:
    skip(SECTION, "latency benchmark", "service unreachable or no frame")
    BENCH = None

---
## 5 · Faces

`embed` needs a real face. Without one the checks are skipped rather than failed
— a grey rectangle containing no face is a correct `404`, not a broken service.

Set `FACE_IMAGE` at the top to exercise the full enroll → embed → match round
trip. The enrollment is removed at the end of the notebook.

In [ ]:
SECTION = "faces"
TEST_LABEL = "_notebook_probe"
FACE_JPEG = None

if FACE_IMAGE:
    try:
        FACE_JPEG = open(os.path.expanduser(FACE_IMAGE), "rb").read()
        info(SECTION, "face image loaded", f"{len(FACE_JPEG) / 1024:.1f} KB from {FACE_IMAGE}")
    except OSError as exc:
        info(SECTION, "face image unreadable", str(exc))

if REACHABLE and FACE_JPEG:
    SEQ += 1
    r = post_frame("/v1/faces/embed", FACE_JPEG, seq=SEQ)
    if r.status_code == 404:
        skip(SECTION, "POST /v1/faces/embed", "no face detected in FACE_IMAGE — try a clearer photo")
    elif check(SECTION, "embed HTTP 200", r.status_code == 200, f"HTTP {r.status_code}: {r.text[:200]}"):
        b = r.json()
        check(SECTION, "seq echoed", b.get("seq") == SEQ)
        check(SECTION, "embedding is 512-d", len(b.get("embedding", [])) == 512,
              f"got {len(b.get('embedding', []))}")
        q = b.get("quality", 0)
        check(SECTION, "quality in range", 0.0 <= q <= 1.0, f"quality={q:.2f}")
        if q < 0.6:
            info(SECTION, "low quality crop",
                 f"{q:.2f} is below the ~0.6 the Pi should require before "
                 f"spending a match call. Fine here; worth knowing.")
        EMBEDDING = b["embedding"]

        # enroll -> match round trip. Matching a face against itself must score
        # ~1.0; anything lower points at a normalisation bug, which otherwise
        # presents as "recognition just doesn't work".
        r = S.post(url("/v1/faces/enroll"),
                   data={"label": TEST_LABEL},
                   files=[("images", ("face.jpg", FACE_JPEG, "image/jpeg"))],
                   timeout=20.0)
        if check(SECTION, "enroll HTTP 200", r.status_code == 200, r.text[:200]):
            info(SECTION, "enrolled", json.dumps(r.json()))

            r = S.post(url("/v1/faces/match"), json={"embedding": EMBEDDING, "top_k": 3}, timeout=10.0)
            if check(SECTION, "match HTTP 200", r.status_code == 200, r.text[:200]):
                b = r.json()
                top = b["matches"][0] if b.get("matches") else None
                check(SECTION, "matches itself", top is not None and top["label"] == TEST_LABEL,
                      json.dumps(b.get("matches")))
                sim = top["similarity"] if top else None
                check(SECTION, "self-similarity ~1.0",
                      sim is not None and sim > 0.95,
                      f"{sim}" if (sim is not None and sim > 0.95) else
                      f"{sim} — a face must match itself at ~1.0. Anything lower "
                      f"points at an embedding normalisation bug, which otherwise "
                      f"presents as 'recognition just doesn't work'.")
                check(SECTION, "threshold returned", "threshold" in b,
                      f"threshold={b.get('threshold')}")
                # Best-score-per-person: one enrolled face must not appear twice.
                labels = [m["label"] for m in b["matches"]]
                check(SECTION, "no duplicate labels in matches",
                      len(labels) == len(set(labels)), str(labels))
elif REACHABLE:
    skip(SECTION, "face endpoints", "set FACE_IMAGE at the top to exercise these")
else:
    skip(SECTION, "face endpoints", "service unreachable")

In [ ]:
SECTION = "faces"
# The no-face path is a normal outcome, and worth proving it behaves. Only
# meaningful against real models — the fake encoder returns an embedding for
# anything large enough, which is the right behaviour for a stub and the wrong
# thing to assert against.
if REACHABLE and FRAME and FRAME_SOURCE == "synthetic" and REAL_BACKEND:
    SEQ += 1
    r = post_frame("/v1/faces/embed", FRAME, seq=SEQ)
    check(SECTION, "no-face crop returns 404 (not an error)", r.status_code == 404,
          f"HTTP {r.status_code} — expected 404 for an image with no face")
elif REACHABLE and not REAL_BACKEND:
    skip(SECTION, "no-face crop returns 404", "only meaningful against real models")

---
## 6 · `POST /v1/depth`

Fallback only. The lidar gives a metric range at the detection's bearing,
locally and for free; this endpoint exists for objects above or below the lidar
plane.

The check that matters is the `metric` flag, not the numbers.

In [ ]:
SECTION = "depth"

if REACHABLE and FRAME:
    SEQ += 1
    points = [[320, 180], [100, 90]]
    r = post_frame("/v1/depth", FRAME, seq=SEQ, points=json.dumps(points), timeout=30.0)
    if check(SECTION, "HTTP 200", r.status_code == 200, f"HTTP {r.status_code}: {r.text[:200]}"):
        b = r.json()
        check(SECTION, "seq echoed", b.get("seq") == SEQ)
        check(SECTION, "one depth per point", len(b.get("depths_m", [])) == len(points),
              f"{len(b.get('depths_m', []))} values for {len(points)} points")
        check(SECTION, "one confidence per point", len(b.get("confidence", [])) == len(points))
        check(SECTION, "metric flag present", isinstance(b.get("metric"), bool),
              f"metric={b.get('metric')}")
        if not b.get("metric"):
            info(SECTION, "values are NOT metres",
                 "relative inverse depth. Branch on the flag, never on the "
                 "depths_m field name — using these as distances misplaces objects.")
        info(SECTION, "depths", f"{b.get('depths_m')} ({b.get('inference_ms', 0):.0f} ms)")
else:
    skip(SECTION, "POST /v1/depth", "service unreachable or no frame")

---
## 7 · The error contract

The Pi has to react differently to each of these, so each one has to actually
happen. A service that returns 500 for everything is one the Pi cannot be
written against.

In [ ]:
SECTION = "errors"

if REACHABLE:
    SEQ += 1
    r = post_frame("/v1/detect", b"this is not a jpeg", seq=SEQ)
    check(SECTION, "garbage body -> 400 image_undecodable",
          r.status_code == 400 and r.json().get("error") == "image_undecodable",
          f"HTTP {r.status_code}: {r.text[:120]}")

    SEQ += 1
    r = post_frame("/v1/detect", b"\x00" * (5 * 1024 * 1024), seq=SEQ, timeout=30.0)
    check(SECTION, "oversized body -> 413 image_too_large",
          r.status_code == 413 and r.json().get("error") == "image_too_large",
          f"HTTP {r.status_code}: {r.text[:120]}")

    if FRAME:
        r = S.post(url("/v1/detect"),
                   files={"image": ("f.jpg", FRAME, "image/jpeg")}, timeout=10.0)
        check(SECTION, "missing seq -> 422", r.status_code == 422, f"HTTP {r.status_code}")

    r = S.post(url("/v1/faces/match"), json={"embedding": []}, timeout=10.0)
    check(SECTION, "empty embedding -> 422", r.status_code == 422, f"HTTP {r.status_code}")

    if FRAME:
        SEQ += 1
        r = post_frame("/v1/depth", FRAME, seq=SEQ, points="not json")
        check(SECTION, "malformed points -> 422", r.status_code == 422, f"HTTP {r.status_code}")

    r = S.delete(url("/v1/faces/_definitely_not_enrolled"), timeout=10.0)
    check(SECTION, "unknown label -> 404", r.status_code == 404, f"HTTP {r.status_code}")
else:
    skip(SECTION, "error contract", "service unreachable")

### Backpressure

There is one GPU, so the service serializes inference and **refuses** rather
than queues once too many requests are waiting. That `503` is the server half of
the drop-don't-queue rule: the Pi treats it as a dropped frame and moves on.

This check is informational — on a fast mini with a small frame the queue may
never fill, and that is a fine outcome.

In [ ]:
SECTION = "errors"

if REACHABLE and FRAME:
    def fire(i):
        try:
            return post_frame("/v1/detect", FRAME, seq=90_000 + i, timeout=30.0).status_code
        except requests.exceptions.RequestException as exc:
            return type(exc).__name__

    with ThreadPoolExecutor(max_workers=12) as pool:
        codes = list(pool.map(fire, range(12)))

    ok = sum(1 for c in codes if c == 200)
    busy = sum(1 for c in codes if c == 503)
    other = [c for c in codes if c not in (200, 503)]

    check(SECTION, "no unexpected failures under load", not other, f"saw {other}")
    if busy:
        info(SECTION, "backpressure engaged",
             f"{ok} served, {busy} refused with 503. Correct — the Pi should "
             f"drop those frames rather than retry them.")
    else:
        info(SECTION, "queue never filled",
             f"all {ok} concurrent calls served. Fine. The Pi only ever has one "
             f"request in flight anyway; this was a deliberate overload.")
else:
    skip(SECTION, "backpressure", "service unreachable or no frame")

---
## 8 · Cleanup and summary

In [ ]:
SECTION = "cleanup"
if REACHABLE and FACE_JPEG:
    r = S.delete(url(f"/v1/faces/{TEST_LABEL}"), timeout=10.0)
    check(SECTION, "test enrollment removed", r.status_code in (200, 404),
          f"HTTP {r.status_code}")
else:
    skip(SECTION, "test enrollment removed", "nothing was enrolled")

In [ ]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
from collections import Counter

counts = Counter(status for _, _, status, _ in RESULTS)
failed = [(s, n, d) for s, n, status, d in RESULTS if status == "FAIL"]

width = 62
print("=" * width)
print("  VISION SERVICE ENDPOINT CHECK")
print("=" * width)
print(f"  target     {BASE_URL}")
print(f"  backend    {(health or {}).get('backend', 'unreachable')}")
print(f"  frame      {FRAME_SOURCE or 'none'}"
      + (f", {len(FRAME) / 1024:.1f} KB" if FRAME else ""))
if BENCH:
    print(f"  latency    p50 {BENCH['p50']:.0f} ms | p95 {BENCH['p95']:.0f} ms "
          f"| p99 {BENCH['p99']:.0f} ms")
print("-" * width)

by_section = {}
for section, name, status, _ in RESULTS:
    by_section.setdefault(section, Counter())[status] += 1
for section, c in by_section.items():
    bits = " ".join(f"{k.lower()} {v}" for k, v in sorted(c.items()))
    print(f"  {section:<14} {bits}")

print("-" * width)
print(f"  {counts['PASS']} passed · {counts['FAIL']} failed · "
      f"{counts['SKIP']} skipped · {counts['INFO']} notes")
print("=" * width)

if failed:
    print("\nFAILURES")
    for section, name, detail in failed:
        print(f"\n  [{section}] {name}")
        if detail:
            for line in str(detail).splitlines():
                print(f"    {line}")
else:
    print("\nEvery check passed.")
    if not REAL_BACKEND:
        print("Note: the service was running the FAKE backend, so the contract "
              "is verified but the detections were invented.")
    if BENCH and BENCH["p99"] < DETECT_TIMEOUT_S * 1000:
        print(f"p99 of {BENCH['p99']:.0f} ms fits inside the "
              f"{DETECT_TIMEOUT_S * 1000:.0f} ms budget. Phase V1's question is "
              f"answered — you can build on these numbers.")

---

### If something failed

| Symptom | Where to look |
|---|---|
| Cannot reach the service | Mini asleep (`caffeinate` is in `run.sh`), service not running, or `BASE_URL` wrong |
| `backend` is `fake` | Start the service with `VS_BACKEND=real`. It is inventing objects |
| p99 over 600 ms | Payload too big (check `DETECT_LONG_EDGE_PX`), wifi contention, or the mini is busy |
| p99 ≫ p50 | Contention, not slowness. Something else is using the radio or the CPU |
| Bandwidth over 60 KB/s | Frame too large. This competes with the voice uplink and audio loses |
| Detect returns nothing | Expected with a synthetic frame. Point the camera at a room |
| Face endpoints skipped | Set `FACE_IMAGE` at the top |

### Next

Re-run this **with the voice session live and the robot driving**. The numbers
above were measured on a quiet Pi; the ones that matter are the ones you get on
a busy one. That comparison is the deliverable of Phase V1, and every constant
in `docs/VISION-DESIGN.md`'s appendix should be re-checked against it.